# Readaloud alignment on a free GPU (Colab or Kaggle)

Step 2 of the readaloud toolkit: runs Qwen3-ForcedAligner on `bundle.zip` made by `prepare.py` (which already did the
transcript, VAD and chunk plan on your computer), and gives you back `results.zip` (`timings.json`) for `build.py`.

**Colab:** *Runtime → Change runtime type → T4 GPU*. Drag `bundle.zip` into the *Files* panel (left sidebar) and wait
until the upload has finished (the file grows in 4 MB steps while uploading; compare its size with the local file),
then *Runtime → Run all*. `results.zip` downloads automatically at the end
(it is also in the Files panel).

**Kaggle (faster: two GPUs):** from your terminal, `uv run kaggle_upload.py WORK` uploads a bundle (as many books
as you like, whenever they're ready) and prints this notebook's link. Here: set `BOOK` at the top of step 2 (empty:
the bundle uploaded last), *Accelerator: GPU T4 x2*, *Run All*, and download `results.zip` from the *Output* panel.
Keep the notebook open for the next book: the GPU session stays yours. Step 2 lists every uploaded bundle; the
chosen one is downloaded with the notebook's own Kaggle login and checked against its fingerprint.

If step 3 is interrupted, just run it again: finished chunks are kept (for as long as the runtime lives).

In [ ]:
# 1. Install the aligner (keeps the platform's own GPU build of PyTorch)
# pip may print 'dependency conflicts' about diffusers / huggingface-hub: harmless, those packages aren't used
!pip install -q qwen-asr==0.0.6 soundfile
import os
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:            # the Kaggle library that kaggle_upload.py uses, to fetch the bundle
    !pip install -q "kaggle @ git+https://github.com/Kaggle/kaggle-cli@9c57dd74c29904c40ff7879eed33644be89d2e2f"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator first")

In [ ]:
# 2. Find the bundle and unpack it into a writable work folder (already-aligned chunks are kept)
import glob, os, shutil, zipfile
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ    # (Colab also has a /kaggle folder)
# on Kaggle everything in /kaggle/working is saved as the run's output: unpack elsewhere, keep only results.zip there
WORK = "/kaggle/temp/work" if ON_KAGGLE else "/content/work"

# Kaggle: which uploaded bundle to align. "" = the one uploaded last; a book's name (its folder name, e.g.
# "oregairu-09") = that book's newest bundle; or a full dataset name such as "readaloud-oregairu-09-4439f6d8"
BOOK = ""

if ON_KAGGLE:
    import hashlib, re
    from kaggle_secrets import UserSecretsClient
    try:                                          # your token, if added under Add-ons -> Secrets
        os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception:
        pass                                      # else the notebook's own Kaggle login
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    user = api.get_config_value("username")
    bundles, seen = [], set()
    for page in range(1, 50):                     # every readaloud-<book>-<sha8> dataset of yours
        got = [d for d in api.dataset_list(mine=True, search="readaloud-", sort_by="updated", page=page) or []
               if d.ref not in seen]
        if not got:
            break
        seen.update(d.ref for d in got)
        bundles += [d for d in got if re.fullmatch(r"readaloud-.+-[0-9a-f]{8}", d.ref.split("/")[-1])]
    bundles.sort(key=lambda d: d.last_updated, reverse=True)
    want = re.sub(r"[^a-z0-9]+", "-", BOOK.lower()).strip("-")
    chosen = next((d for d in bundles if not want or d.ref.split("/")[-1] == want
                   or d.ref.split("/")[-1].rsplit("-", 1)[0] == "readaloud-" + want), None)
    for d in bundles:
        print(("-> " if d is chosen else "   ") + d.ref.split("/")[-1], "  uploaded", str(d.last_updated)[:16])
    if chosen is None:
        raise SystemExit(f"no uploaded bundle {'for ' + repr(BOOK) if BOOK else 'yet'} - run kaggle_upload.py "
                         "first, or set BOOK to one of the names above")
    slug = chosen.ref.split("/")[-1]
    DL = "/kaggle/temp/download"
    fingerprint = slug.rsplit("-", 1)[1]
    done = os.path.join(WORK, ".bundle")
    if not os.path.exists(done) or open(done).read() != fingerprint:
        shutil.rmtree(WORK, ignore_errors=True)   # another bundle's files (tracks, decoded audio) must not mix in
        print(f"downloading {slug} ...", flush=True)
        shutil.rmtree(DL, ignore_errors=True)       # a single file: the whole-dataset archive is built minutes later
        api.dataset_download_file(f"{user}/{slug}", "bundle.bin", DL, force=True, quiet=True, unzip=True)
        h = hashlib.sha256()
        with open(f"{DL}/bundle.bin", "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        if not h.hexdigest().startswith(fingerprint):
            raise SystemExit(f"the downloaded bundle doesn't match {slug} - run this cell again")
        zipfile.ZipFile(f"{DL}/bundle.bin").extractall(WORK)
        open(done, "w").write(fingerprint)
        shutil.rmtree(DL, ignore_errors=True)
    print("bundle", slug)
else:
    # drag bundle.zip into the Files panel (left sidebar), or put it in Google Drive and mount Drive there
    found = glob.glob("/content/bundle.zip") + glob.glob("/content/drive/MyDrive/**/bundle.zip", recursive=True)
    if not found:
        raise SystemExit("No bundle.zip in /content or Google Drive - upload it via the Files panel, wait until the upload has finished, then run this cell again")
    try:
        zipfile.ZipFile(found[0]).extractall(WORK)
    except zipfile.BadZipFile:
        raise SystemExit(f"{found[0]} is incomplete or damaged ({os.path.getsize(found[0]):,} bytes) - "
                         "if the upload is still running, wait for it; otherwise delete it and upload again")
print(sorted(os.listdir(WORK)))

In [ ]:
# 3. Precise alignment on the GPU in float16 (~8 min for a 15 h book on one T4, both GPUs used on Kaggle's T4 x2;
#    re-run this cell to resume after an interruption)
!cd {WORK} && python scripts/align.py . 2>&1 | grep --line-buffered -v -i warn

In [ ]:
# 4. Package the result
OUT = "/kaggle/working/results.zip" if ON_KAGGLE else "/content/results.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, "timings.json"), "timings.json")
    z.write(os.path.join(WORK, "units.json"), "units.json")      # the aligner's raw output, for diagnosis
print("wrote", OUT, os.path.getsize(OUT) // 1024, "KB")
if not ON_KAGGLE:
    from google.colab import files
    files.download(OUT)